# Active-Learning Preprocessing Workflow

This notebook is the only human-facing entrypoint for the PAPI active-learning data loop. Helper scripts remain internal implementation details called from this notebook.

## Rules

- Use YOLOv26m for the current model cycle.
- Keep normal `WideCamera` `5280x3956` data separate from zoom/cropped data.
- Manual corrections are source of truth. Latest non-empty manual label wins.
- Predictions only fill gaps and must use confidence `>= 0.5` unless this notebook records a different reason.
- CVAT upload uses two matched archives per `batch_###`: create the task from `images.zip`, then import `annotations.zip` as Ultralytics YOLO Detection 1.0. Do not use `dataset.zip`, flat filename zips, or legacy annotation-only artifacts.

In [ ]:
from __future__ import annotations

import csv
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import yaml

REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
PYTHON = REPO / ".venv" / "Scripts" / "python.exe"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

DATA = REPO / "data"
MANUAL_CORRECTIONS = DATA / "annotations" / "manual_corrections"
SEQUENCE_DATASET = DATA / "datasets" / "papi_lamp_sequences"
YOLO_DATASET = SEQUENCE_DATASET / "daytime"
VALIDATION_SUMMARY = SEQUENCE_DATASET / "validation_summary.json"
BASE_MODEL = REPO / "yolov26m.pt"
RUN_NAME = "yolov26m_seed1800_batch001_006_corrected_day_cuda"
BEST_MODEL = REPO / "runs" / "papi" / RUN_NAME / "weights" / "best.pt"
CURRENT_MANUAL_SOURCES = [
    MANUAL_CORRECTIONS / "batch_001_corrected_300",
    MANUAL_CORRECTIONS / "batch_002_corrected_300",
    MANUAL_CORRECTIONS / "batch_003_corrected_300",
    MANUAL_CORRECTIONS / "batch_004_corrected_300",
    MANUAL_CORRECTIONS / "batch_005_corrected_300",
    MANUAL_CORRECTIONS / "batch_006_corrected_300",
]
SEED_INDEX_SOURCE = MANUAL_CORRECTIONS / "batch_001_002_003_004_005_006_corrected_1800"

def run(*args: str) -> None:
    cmd = [str(PYTHON), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

print("repo", REPO)
print("python", PYTHON)
print("dataset", YOLO_DATASET)
print("best model", BEST_MODEL)

## 1. Folder Sanity Check

In [ ]:
expected_roots = {"raw", "interim", "labels", "annotations", "cvat", "datasets", "README.md"}
present = {p.name for p in DATA.iterdir()}
unexpected = sorted(present - expected_roots)
print("data roots:", sorted(present))
assert not unexpected, f"Unexpected generated folders/files at data root: {unexpected}"

## 2. Manual Correction Inventory

In [ ]:
manual_sources = [source for source in CURRENT_MANUAL_SOURCES if source.exists()]
for source in manual_sources:
    labels_dir = source / "labels" / "train"
    label_files = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []
    non_empty = sum(1 for p in label_files if p.read_text(encoding="utf-8").strip())
    print(source.name, {"train_txt": (source / "train.txt").exists(), "labels": len(label_files), "non_empty": non_empty})
print("seed index source", SEED_INDEX_SOURCE, SEED_INDEX_SOURCE.exists())


## 3. Rebuild Daytime Seed Dataset

Run this after adding a new corrected CVAT export. The helper is internal; this notebook is the workflow surface. Daytime and nighttime batches stay separate.

In [ ]:
print("Seed datasets are now built from data/datasets/papi_lamp_sequences/.")
print("Do not rebuild deprecated data/datasets/yolo seed folders unless you need historical traceability.")


## 4. Fix And Validate YOLO Dataset

In [ ]:
data_yaml_path = YOLO_DATASET / "data.yaml"
data_yaml = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
assert data_yaml["names"] == {0: "papi_light_red", 1: "papi_light_white"}

allowed = {0, 1}
summary = {"images": 0, "labels": 0, "objects": 0, "bad": 0}
for rel in [line.strip() for line in (YOLO_DATASET / "train.txt").read_text(encoding="utf-8").splitlines() if line.strip()]:
    image = YOLO_DATASET / rel
    label = image.parent.parent / "labels" / f"{image.stem}.txt"
    if not image.exists():
        summary["bad"] += 1
        print("missing image", image)
        continue
    if not label.exists():
        summary["bad"] += 1
        print("missing label", label)
        continue
    summary["images"] += 1
    summary["labels"] += 1
    for line in label.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split()
        summary["objects"] += 1
        if len(parts) != 5 or int(parts[0]) not in allowed or any(not (0 <= float(v) <= 1) for v in parts[1:]):
            summary["bad"] += 1
            print("bad row", label, line)
print(json.dumps(summary, indent=2))
assert summary["bad"] == 0


## 5. YOLOv26m CUDA Smoke Check And Training

`yolov26m.pt` is loaded exactly. If it is missing, Ultralytics will download it. Set `RUN_TRAIN=True` only when ready to fine-tune.

In [ ]:
model = None
try:
    import torch
    from ultralytics import YOLO

    print("cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device", torch.cuda.get_device_name(0))
    model = YOLO(str(BASE_MODEL) if BASE_MODEL.exists() else "yolov26m.pt")
    if not BASE_MODEL.exists() and (REPO / "yolov26m.pt").exists():
        shutil.copy2(REPO / "yolov26m.pt", BASE_MODEL)
    print("loaded", BASE_MODEL if BASE_MODEL.exists() else "yolov26m.pt")
except Exception as exc:
    print("YOLOv26m smoke check failed:", repr(exc))
    print("Fix the Torch/Ultralytics environment before setting RUN_TRAIN=True.")

In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    assert model is not None, "Run and pass the YOLOv26m smoke check before training."
    model.train(
        data=str(YOLO_DATASET / "data.yaml"),
        project=str(REPO / "runs" / "papi"),
        name=RUN_NAME,
        imgsz=1280,
        batch=2,
        epochs=100,
        patience=20,
        device=0,
        amp=True,
        exist_ok=True,
    )
else:
    print("Training disabled. Set RUN_TRAIN=True to fine-tune YOLOv26m.")

## 6. Assisted Annotation With YOLOv26m

Run after `BEST_MODEL` exists. Predictions are confidence-filtered at `0.5` and only fill frames without manual labels.

In [ ]:
# Recreate CVAT batches from the canonical sequence dataset only when a new manual review round is needed.
print("Canonical sequence dataset:", SEQUENCE_DATASET)
print("Latest validation summary:", VALIDATION_SUMMARY)


## 7. Build CVAT Batches

In [ ]:
print("Canonical day/night sequence datasets are materialized under data/datasets/papi_lamp_sequences/.")
print("Use the validation cell below before training or video reconstruction; old batch-oriented CVAT folders were removed during cleanup.")


## 8. Validate CVAT `images.zip` / `annotations.zip` Pairs

In [ ]:
summary = {}
errors = []
for regime_root in [SEQUENCE_DATASET / "daytime", SEQUENCE_DATASET / "nighttime"]:
    manifest = json.loads((regime_root / "manifest.json").read_text(encoding="utf-8"))
    entries = [line.strip() for line in (regime_root / "train.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
    images = 0
    labels = 0
    objects = 0
    for rel in entries:
        image = regime_root / rel
        label = image.parent.parent / "labels" / f"{image.stem}.txt"
        if not image.exists():
            errors.append((regime_root.name, "missing_image", rel))
            continue
        if not label.exists():
            errors.append((regime_root.name, "missing_label", str(label)))
            continue
        images += 1
        labels += 1
        for line in label.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            parts = line.split()
            objects += 1
            if len(parts) != 5 or parts[0] not in {"0", "1"}:
                errors.append((regime_root.name, "bad_label", str(label), line))
    summary[regime_root.name] = {"images": images, "labels": labels, "objects": objects, "manifest": manifest["totals"]}
print(json.dumps(summary, indent=2))
assert not errors, errors[:20]


## 9. Quality Gates

In [ ]:
run("-m", "pytest")
run("-m", "ruff", "check", "src", "tests", "scripts")

## 10. BigBrain Update Checklist

After a real training or annotation cycle, record: model base (`yolov26m.pt`), checkpoint path, dataset path, CVAT batch counts, validation counts, CVAT import result, and any open labeling issues.